In [ ]:
import torch
import deepwave

# 1. Strict Device Allocation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    print("WARNING: Executing on CPU. HPC tensor parallelization will be severely bottlenecked.")
else:
    print(f"HPC Node Active: {torch.cuda.get_device_name(0)}\n")

# 2. High-Resolution Baseline Parameters
nx, ny = 70, 70
dx = 10.0
nt = 1000
dt = 0.004
num_shots = 5  

# 3. Velocity Macro-Model (The Neural Network Proxy)
# [Rasht-Behesht 2022 Compliance]: Setting requires_grad=True is the foundational 
# requirement to compute the spatial derivatives for the Physics-Informed Regularization.
# [Rasht-Behesht 2022 Adaptation]: For this Step 1 stress test, we are optimizing a 
# dense discrete grid directly. In the final architecture (Step 2), this matrix will 
# be replaced by an MLP mapping spatial coordinates (x, z) to v(x, z).
v = torch.full((nx, ny), 1500.0, device=device, requires_grad=True)

# 4. Dense Acquisition Geometry Tensors
source_locations = torch.zeros(num_shots, 1, 2, device=device)
source_locations[:, 0, 0] = 0.0 
source_locations[:, 0, 1] = torch.linspace(dx * 10, (nx * dx) - (dx * 10), num_shots).to(device)

receiver_locations = torch.zeros(num_shots, nx, 2, device=device)
receiver_locations[:, :, 0] = 0.0 
receiver_locations[:, :, 1] = (torch.arange(nx) * dx).repeat(num_shots, 1).to(device)

# 5. Source Wavelet Matrix (15Hz Ricker)
freq = 15.0
source_amplitudes = (
    deepwave.wavelets.ricker(freq, nt, dt, 1.5 / freq)
    .reshape(1, 1, -1)
    .repeat(num_shots, 1, 1)
    .to(device)
)

print(f"Dispatching full {num_shots}-shot tensor block to PyTorch Autograd Engine...")

# 6. Simultaneous Forward Propagation (Zero Python Overhead)
# [Rasht-Behesht 2022 Adaptation]: The authors used purely neural networks (PINNs) 
# to solve the forward wave equation via collocation points. We explicitly diverge here. 
# We use Deepwave (finite-difference solvers) to compute the transient wavefield u(t, x, z). 
# This Hybrid DDR-PINN approach bypasses the spectral bias and massive memory bottlenecks 
# pure PINNs face with high-frequency seismic data.
out = deepwave.scalar(
    v, grid_spacing=dx, dt=dt,
    source_amplitudes=source_amplitudes,
    source_locations=source_locations,
    receiver_locations=receiver_locations
)

print("Executing Adjoint State Backpropagation...")

# 7. Global Gradient Calculation
# [Rasht-Behesht 2022 Compliance]: The gradient computed here via Autograd directly mirrors 
# their calculation of the data mismatch gradient. This will later be coupled with the 
# PDE residual to enforce the physical laws during the optimization loop.
loss = out[-1].sum()
loss.backward() 

print("\nArchitecture Validated: Full computational graph successfully processed.")
print(f"Gradient Tensor Shape Ready for Optimizer: {v.grad.shape}")